# Training Mechanics and Evaluation

Goal: Practice model evaluation, metrics and hyperparameter tuning on medical classification.

In [3]:
import sys
sys.path.append("../")  # adds parent folder to Python path
from workshop_utils import load_dataset, basic_train_test_split, build_preprocessor
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score

df = load_dataset("cvd")
X_train, X_test, y_train, y_test = basic_train_test_split(df, target="CVD")

num_features = ["age", "chol", "bp"]
cat_features = []
preprocessor = build_preprocessor(num_features, cat_features)

base_pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", MLPClassifier(max_iter=500)),
    ]
)

param_grid = {
    "model__hidden_layer_sizes": [(32,), (64,), (32, 16)],
    "model__alpha": [0.001, 0.0001],
}

search = GridSearchCV(base_pipe, param_grid=param_grid, cv=5)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)

y_pred = search.predict(X_test)
print(classification_report(y_test, y_pred))

if hasattr(search.best_estimator_["model"], "predict_proba"):
    y_prob = search.predict_proba(X_test)[:, 1]
    print("ROC AUC:", roc_auc_score(y_test, y_prob))

/home/jislam/.local/lib/python3.10/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Best params: {'model__alpha': 0.001, 'model__hidden_layer_sizes': (32,)}
              precision    recall  f1-score   support

           0       0.97      1.00      0.99        78
           1       0.00      0.00      0.00         2

    accuracy                           0.97        80
   macro avg       0.49      0.50      0.49        80
weighted avg       0.95      0.97      0.96        80

ROC AUC: 0.1282051282051282


/home/jislam/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/jislam/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/jislam/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
